<a href="https://colab.research.google.com/github/Jlnlys/machine-learning-from-scratch/blob/main/NaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

**NAIVE BAYES**

Bayes theorem

$P(A | B) = \frac{P(B|A) * P(A)}{P(B)}$

In our case:

$P(y|X) = \frac{P(X|y) *p(y)}{P(X)}$

Whith vector feature X:

$X = {(x1,x2,...,x_n)}$

So we assume all features are mutally independent:

$P(x_1|y)*P(x_2|y)*...*P(x_n|y) * P(y)$

In [ ]:
class My_NB():
  def __init__(self):
    self.X = None
    self.y = None
    self.proba_true = None
    self.proba_false = None

  def fit(self,X,y):
    self.X = X
    self.y =y
    self.proba_true = len(self.X[self.y==1])/len(self.y)
    self.proba_false =len(self.X[self.y==0])/len(self.y)
    self.mean_true = self.X[self.y==1].mean(axis=0)
    self.std_true = self.X[self.y==1].std(axis=0)
    self.mean_false = self.X[self.y==0].mean(axis=0)
    self.std_false = self.X[self.y==0].std(axis=0)


  def gaussian(self,t,mean,std):
    return (1 / (np.sqrt(2 * np.pi) * std)) *  np.exp(-((t - mean) ** 2) / (2 * std ** 2))

  def predict(self,X):
    y_pred = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
      sum_true = self.proba_true
      sum_false = self.proba_false
      for c in range(X.shape[1]):
        actu = X[i][c]
        mi_true = self.gaussian(actu,self.mean_true[c],self.std_true[c])
        mi_false = self.gaussian(actu,self.mean_false[c],self.std_false[c])
        sum_true *= mi_true
        sum_false *= mi_false
      if(sum_true > sum_false):
        y_pred[i] = 1
      else:
        y_pred[i] = 0
    return y_pred


In [ ]:
import pandas as pd

In [ ]:
cols = ["fLength","fWidth","fSize","fConc","fConc1","fAsym","fM3Long","fM3Trans","fAlpha","fDist","class"]
df = pd.read_csv("magic04.data",names=cols)
df.head()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g


In [ ]:
df["class"] = (df["class"] == "g").astype(int)
df.head()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,1
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,1
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,1
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,1
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,1


In [ ]:
train, valid, test = np.split(df.sample(frac=1),[int(0.6*len(df)),int(0.8*len(df))])

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler

In [ ]:
def standardize(dataframe,random=True):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values
  scaler = StandardScaler()
  x = scaler.fit_transform(x)
  if random:
    x,y = RandomOverSampler().fit_resample(x,y)

  data = np.hstack((x,np.reshape(y,(-1,1))))

  return data,x,y

In [ ]:
train_data, train_x, train_y = standardize(train,True)
valid_data, valid_x, valid_y = standardize(valid,False)
test_data, test_x, test_y = standardize(test,False)

In [ ]:
from sklearn.metrics import  classification_report
from sklearn.naive_bayes import GaussianNB

In [ ]:
my_nb = My_NB()
my_nb.fit(train_x,train_y)
y_pred = my_nb.predict(test_x)
print(classification_report(test_y,y_pred))


              precision    recall  f1-score   support

           0       0.69      0.40      0.50      1315
           1       0.74      0.91      0.81      2489

    accuracy                           0.73      3804
   macro avg       0.71      0.65      0.66      3804
weighted avg       0.72      0.73      0.71      3804



SKLEARN

In [ ]:
nb = GaussianNB()
nb.fit(train_x,train_y)
y_pred = nb.predict(test_x)
print(classification_report(test_y,y_pred))

              precision    recall  f1-score   support

           0       0.69      0.40      0.50      1315
           1       0.74      0.91      0.81      2489

    accuracy                           0.73      3804
   macro avg       0.71      0.65      0.66      3804
weighted avg       0.72      0.73      0.71      3804



That's it for Naive Bayes! I implemented it from scratch to really understand how it works under the hood, instead of just calling .fit(). I compared it against scikit-learn's GaussianNB and got the same results, which confirms it works correctly.
More algorithms from scratch coming soon!
Feel free to reach out
 Julien
 https://github.com/Jlnlys